# 옷장패션 데이터 정제 보고서

## 1. 데이터 개요

옷장패션 주문 데이터는 1,500행 × 8열이며, 과제 양식의 '7열'은 return_amount를 빠뜨린 표기다.

| 컬럼 | 내용 |
| --- | --- |
| order_id | 주문번호 (K00001~K01500) |
| customer_age | 고객 나이 |
| category | 상의·하의·신발·액세서리 |
| channel | web / app |
| price | 단가 (15,900~129,900원, 5종) |
| quantity | 수량 |
| amount | 결제 금액 = price × quantity |
| return_amount | 환불 금액 (0 = 환불 없음) |

## 2. 진단 결과

결측은 amount 51건(3.40%)과 price 5건(0.33%)뿐이고, 명백한 오류값은 나이 5건·수량 1건·금액 1건이다.

| 항목 | 대상 | 건수 | 비율 | 비고 |
| --- | --- | --- | --- | --- |
| 결측 | amount | 51건 | 3.40% | app 채널에 쏠림 |
| 결측 | price | 5건 | 0.33% | 접속 시점 가격 누락 |
| 이상치 | customer_age = 999·0 | 5건 | 0.33% | 999 3건, 0 2건 |
| 이상치 | quantity = 200 | 1건 | 0.07% | K00025 |
| 이상치 | amount = 50,000,000원 | 1건 | 0.07% | K00548 |

**결측 유형: amount는 MAR.** 결측 비율이 app 4.95%(909건 중 45건), web 1.02%(591건 중 6건)로 channel에 따라 다르다.

**IQR(k=1.5) 결과는 그대로 쓸 수 없다.**

- customer_age: 18건이 걸리지만 13건은 5~12세·54~60세의 자연스러운 분포 꼬리다.
- amount: 145건이 걸리지만 144건은 259,800원·389,700원(129,900원 × 2·3개)의 정상 고가 주문이다.
- return_amount: 93%가 0이라 Q1 = Q3 = 0, IQR = 0이 되어 환불 122건이 전부 이상치로 잡힌다. IQR 적용 불가.

**가격 × 수량 교차 검증에서 두 극단값 모두 계산이 맞지 않는다.**

| 주문 | price | quantity | amount | price × quantity |
| --- | --- | --- | --- | --- |
| K00025 | 29,900 | 200 | 89,700 | 5,980,000 |
| K00548 | 49,900 | 1 | 50,000,000 | 49,900 |

K00025는 금액이 3개 기준이라 수량 입력 오류, K00548은 1개 주문이라 금액 입력 오류 가능성이 있다.

## 3. 처리 결정과 근거

원본 값은 지우지 않고, 바꾼 칸마다 플래그 열을 남겨 언제든 되돌릴 수 있게 했다.

| 이슈 | 결정 | 근거 | 한계 |
| --- | --- | --- | --- |
| amount 결측 51건 | 채널별 중앙값 대체 + amount_imputed 플래그 | MAR 가설 부합 | 두 채널 중앙값이 모두 79,900원이라 결과는 전체 중앙값 대체와 같음. 대체값이 한 값에 몰려 분산 과소평가 |
| price 결측 5건 | amount ÷ quantity로 복원 + price_restored 플래그 | 금액 = 가격 × 수량 규칙. 복원값 5건 모두 실제 가격표에 존재 | amount·quantity가 정상일 때만 가능 |
| customer_age 999·0 (5건) | 결측 처리 후 중앙값(33세) 대체 + age_imputed 플래그 | 사람 나이로 불가능한 값 = 입력 오류 | 실제 나이는 알 수 없음. 5~12세 주문은 자녀용 구매일 수 있어 유지 |
| quantity 200 (1건) | 값 유지 + quantity_outlier 플래그 | 금액(89,700원)이 3개 기준이라 수량 입력 오류 의심, 확정 불가 | 수량 합계 분석 시 이 건 제외 필요 |
| amount 5천만 원 (1건) | 값 유지 + amount_outlier 플래그 | 도매 의심이지만 49,900원 × 1개로 계산 불일치 → 입력 오류 가능성 | 매출 합계·평균 왜곡. 포함·제외 두 버전 비교 필요 |
| return_amount 0 | 처리 안 함 | 0은 '환불 없음'이라는 실제 값, 결측 아님 | — |

## 4. 처리 후 검증

처리 후 결측은 0건이고, 원본 1,500행은 모두 보존되었다.

| 점검 항목 | 결과 |
| --- | --- |
| 결측 | 0건 (모든 열) |
| customer_age 범위 | 5 ~ 60세 (오류값 제거, 저연령 주문은 업무 확인 필요) |
| 복원한 price | 5건 모두 실제 가격표(15,900·29,900·49,900·79,900·129,900원)와 일치 |
| amount_outlier 플래그 | 145건 보존 (IQR k=1.5 기준, 5천만 원 1건 포함. k=3이면 50건) |
| quantity_outlier 플래그 | 1건 보존 |
| app 평균 금액 | 158,965원 → 155,051원 |
| web 평균 금액 | 99,723원 → 99,522원 |

결측이 많은 app의 평균이 더 크게 내려가, 결측이 몰린 그룹일수록 대체 방법의 영향을 크게 받는다.

## 5. 후속 권고

- [ ] K00548(5천만 원)·K00025(수량 200) 주문 원본을 원천 시스템에서 확인
- [ ] 도매 가능 고객 식별을 위해 customer_id 단위 과거 구매 이력 확보
- [ ] amount 결측이 app에 쏠린 원인(앱 로그 수집 오류 등)을 개발팀에 확인
- [ ] 정기 점검에 IQR과 함께 price × quantity = amount 규칙 검증 포함

In [2]:
# ═════════════════════════════════════════════
# 0. 준비 + 데이터 생성
# ═════════════════════════════════════════════
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)

np.random.seed(11)
n = 1500

partner = pd.DataFrame({
    "order_id": [f"K{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(33, 8, n).round().astype(int),
    "category": np.random.choice(["상의", "하의", "신발", "액세서리"], n, p=[0.35, 0.3, 0.2, 0.15]),
    "channel": np.random.choice(["web", "app"], n, p=[0.4, 0.6]),
    "price": np.random.choice([15900, 29900, 49900, 79900, 129900], n),
    "quantity": np.random.choice([1, 1, 1, 2, 2, 3], n),
})
partner["amount"] = partner["price"] * partner["quantity"]
partner["return_amount"] = np.where(
    np.random.rand(n) < 0.07, partner["amount"] * np.random.uniform(0.5, 1.0, n), 0
).round(0)

partner.loc[partner.sample(3, random_state=1).index, "customer_age"] = 999
partner.loc[partner.sample(2, random_state=2).index, "customer_age"] = 0
app = partner["channel"] == "app"
partner.loc[partner[app].sample(frac=0.05, random_state=3).index, "amount"] = np.nan
partner.loc[partner[~app].sample(frac=0.01, random_state=4).index, "amount"] = np.nan
partner.loc[partner.sample(5, random_state=5).index, "price"] = np.nan
partner.loc[partner.sample(1, random_state=6).index, "quantity"] = 200
partner.loc[partner.sample(1, random_state=7).index, "amount"] = 50_000_000

print("옷장패션 데이터 준비 완료:", partner.shape)

# ═════════════════════════════════════════════
# 1. 데이터 개요
# ═════════════════════════════════════════════
print("\n[1. 데이터 개요]")
print("행/열:", partner.shape)                 # (1500, 8)
print("주요 컬럼:", partner.columns.tolist())

# ═════════════════════════════════════════════
# 2-1. 결측 비율
# ═════════════════════════════════════════════
def missing_summary(df):
    s = df.isnull().sum()
    p = (df.isnull().mean() * 100).round(2)
    out = pd.DataFrame({"missing": s, "missing_pct(%)": p})
    return out[out["missing"] > 0].sort_values("missing", ascending=False)

print("\n[2-1. 결측 비율]")
display(missing_summary(partner))             # amount 3.40%, price 0.33%

# ═════════════════════════════════════════════
# 2-2. 이상치 건수
# ═════════════════════════════════════════════
print("\n[2-2. 이상치 건수]")
print("customer_age 999·0:", partner["customer_age"].isin([999, 0]).sum(), "건")   # 5
print("quantity 200      :", (partner["quantity"] == 200).sum(), "건")             # 1
print("amount 50,000,000 :", (partner["amount"] == 50_000_000).sum(), "건")         # 1

# ═════════════════════════════════════════════
# 2-3. IQR 점검
# ═════════════════════════════════════════════
def detect_outliers_iqr(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, (lower, upper)

print("\n[2-3. IQR 점검]")
for col in ["customer_age", "quantity", "amount", "return_amount"]:
    m, (lo, up) = detect_outliers_iqr(partner[col])
    print(f"{col:14s} IQR 이상치 {m.sum():4d}건 | 경계 {lo:,.1f} ~ {up:,.1f}")

# ═════════════════════════════════════════════
# 2-4. 결측 유형 (MAR 근거)
# ═════════════════════════════════════════════
print("\n[2-4. 채널별 amount 결측]")
print(pd.crosstab(partner["channel"], partner["amount"].isna(), margins=True))
print((partner.groupby("channel")["amount"].apply(lambda s: s.isna().mean()) * 100).round(2))
# app 4.95%, web 1.02%

# ═════════════════════════════════════════════
# 2-5. 가격 × 수량 교차 검증
# ═════════════════════════════════════════════
print("\n[2-5. 가격 × 수량 ≠ 금액인 주문]")
calc = partner["price"] * partner["quantity"]
broken = partner["amount"].notna() & partner["price"].notna() & (calc != partner["amount"])
display(partner.loc[broken, ["order_id", "price", "quantity", "amount"]].assign(price_x_qty=calc[broken]))

# ════════════════════════════════════════════

옷장패션 데이터 준비 완료: (1500, 8)

[1. 데이터 개요]
행/열: (1500, 8)
주요 컬럼: ['order_id', 'customer_age', 'category', 'channel', 'price', 'quantity', 'amount', 'return_amount']

[2-1. 결측 비율]


,missing,missing_pct(%)
amount,51,3.40
price,5,0.33



[2-2. 이상치 건수]
customer_age 999·0: 5 건
quantity 200      : 1 건
amount 50,000,000 : 1 건

[2-3. IQR 점검]
customer_age   IQR 이상치   18건 | 경계 12.6 ~ 53.6
quantity       IQR 이상치    1건 | 경계 -0.5 ~ 3.5
amount         IQR 이상치  145건 | 경계 -75,600.0 ~ 253,200.0
return_amount  IQR 이상치  122건 | 경계 0.0 ~ 0.0

[2-4. 채널별 amount 결측]
amount   False  True   All
channel                   
app        864    45   909
web        585     6   591
All       1449    51  1500
channel
app    4.95
web    1.02
Name: amount, dtype: float64

[2-5. 가격 × 수량 ≠ 금액인 주문]


,order_id,price,quantity,amount,price_x_qty
24,K00025,29900.0,200,89700.0,5980000.0
547,K00548,49900.0,1,50000000.0,49900.0
